# Script Outline



## Prepare Workspace

#### Import Packages

Need to factor in tract IDs changing over time?

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

#### File paths

In [ ]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

#### User Defined Functions/Objects

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs for Importing ACS Data

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Parameters')
df_params = df_params[df_params['Run'] == 1]

# Set parameters for querying ACS data
estimate    = df_params['estimate' ].values[0]
sample_type = df_params['sample'   ].values[0]
geography   = df_params['geography'].values[0]

inputs = sample_type + '_' + geography
proportions = df_params['proportions'].values[0]
population_weights = df_params['population_weights'].values[0]

print(estimate)
print(sample_type)
print(geography)
print(inputs)
print("Proportions?: " + proportions)
print("Population weights?: " + population_weights)

#### Import ACS tables/variables mapping and FIPS mapping

In [ ]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
df_inputs = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = inputs, usecols = 'A:H')

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'    ].values[0])
year_end   = int(df_inputs['year_end'      ].values[0])
report_theme  =  df_inputs['report_theme'  ].values[0]
sp_folder_out =  df_inputs['sp_folder'     ].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set years
years_to_import = list(range(year_start, year_end+1))



if sample_type == 'DEC':

    # Reset years to import for DEC
    years_to_import = [2000, 2010, 2020]
    
    # Set DEC variables
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    # Import County FIPS mapping
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    
    # Convert to dictionary
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)
    


# For ACS1 or ACS5 tables
if sample_type == 'ACS':

    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except:
            pass
            
    # Set tables and variables to import
    list_vars = ['NAME'] + df_vars['ID'].to_list()
    tables = df_vars['Table'].unique()

    # For tract and county level pull
    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
        
        # Import County FIPS mapping
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
        
        # Convert to dictionary
        dict_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values))
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(tables)
        print(list_vars)

    
    # For MSA level pull
    if inputs == 'ACS_MSA':
    
        # Set MSA
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)




# For PUMS1 or PUMS5 tables
if inputs == 'PUMS_PUMA':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS??)
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except:
            pass  

    # Set PUMA variables
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['PUMA'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
        

    # Import County FIPS mapping
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'State FIPS': object, 'County FIPS': object, 'TRACTCE': object, 'PUMA5CE': object})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    
    # Convert to dictionary
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values)) 
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    print(dict_fips)
    print(dict_vars)


# view
print(report_theme)
print(sp_folder_out)
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

## Import Data

#### Create Census Tracts Table

#### Import ACS Data by Census Tracts

In [ ]:
# For ACS1 or ACS5 tables
if sample_type == 'ACS':

    ## Build initial ID fields ##
    print("Building initial ID fields")
    print("")
          
    # initialize empty list to store data frames
    list_df_acs = []
    
    # only want one table
    df_table = df_vars[df_vars['Table'] == tables[0]]
    list_table_vars = ['NAME'] + df_table['ID'].to_list()[0:2]
    variables = ",".join(list_table_vars)
    
    # For tract or county level pull
    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
        # pull data, subset to just ID fields
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    temp = query_acs(api_Key     = api_key
                                     , estimate  = estimate
                                     , geography = geography
                                     , variables = variables
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])

                    if inputs == 'ACS_Tract':
                        temp = temp[['NAME', 'state', 'county', 'tract', 'Year']]
                    if inputs == 'ACS_County':
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                    list_df_acs.append(temp)
                except:
                    pass
                    

    # For MSA level pull
    if inputs == 'ACS_MSA':
        # pull all years and counties for each table
        for year in tqdm(years_to_import):
            try:
                temp =  query_acs(api_Key     = api_key
                                  , estimate  = estimate
                                  , geography = geography
                                  , variables = variables
                                  , year      = year
                                  , msa       = msa_to_import)
                
                temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
                list_df_acs.append(temp)
            except:
                pass
                
    # combine all years and counties
    df_acs_raw = pd.concat(list_df_acs)

    # For tracts or counties only
    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
        # merge county name onto table
        df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
        df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
    
        
    print("Finished!")
    print("")
    
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling data from source")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    list_df_acs = []
    
    # iterate through each table
    for table in tables:
    
        # keep track of tables being imported
        print("")
        print("Table ID: " + table)
        list_df_tables = []
    
        # only want one table
        df_table = df_vars[df_vars['Table'] == table]
        list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+25] for x in range(0, len(df_table['ID'].to_list()), 25)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        # pull all years and counties for each table
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_tables = []
                
            for year in tqdm(years_to_import):
                try:

                    # Tract level inputs
                    if (inputs == 'ACS_Tract') | (inputs == 'ACS_County'):
                        list_df_tables.append(
                            query_acs(api_Key     = api_key
                                      , estimate  = estimate
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , state     = state
                                      , county    = dict_fips[state])
                        )

                    # MSA level inputs
                    if inputs == 'ACS_MSA':
                        list_df_tables.append(
                            query_acs(api_Key     = api_key
                                      , estimate  = estimate
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , msa       = msa_to_import)
                        ) 
                            
                except:
                    pass
    
            # combine all years and counties
            df_temp = pd.concat(list_df_tables)
    
            # left join data onto key for each geography type
            if inputs == 'ACS_Tract':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'left')
            if inputs == 'ACS_County':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')
            if inputs == 'ACS_MSA':
                df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')

    
    print("Finished!")


if sample_type == 'DEC':

    ## Build initial ID fields ##
    print("Building initial ID fields")
    print("")
          
    # initialize empty list to store data frames
    list_df_acs = []
    
    # For tract level pull
    if (inputs == 'DEC_Tract') | (inputs == 'DEC_County'):
        # pull data, subset to just ID fields
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    temp = query_acs(api_Key     = api_key
                                     , estimate  = estimate
                                     , geography = geography
                                     , variables = ','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                    
                    if inputs == 'DEC_Tract':
                        temp = temp[['NAME', 'state', 'county', 'tract', 'Year']]
                    if inputs == 'DEC_County':
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                    list_df_acs.append(temp)   
                except:
                    pass

    # For tract level pull
    if inputs == 'DEC_MSA':
        for year in tqdm(years_to_import):
            try:
                temp = query_acs(api_Key     = api_key
                                 , estimate  = estimate
                                 , geography = geography
                                 , variables = ','.join(dict_vars[str(year)])
                                 , year      = year
                                 , state     = state
                                 , msa       = msa_to_import)
                 
                temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
                list_df_acs.append(temp)  
            except:
                pass

    # combine all years and counties
    df_acs_raw = pd.concat(list_df_acs)

    # For tracts or counties only
    if (inputs == 'DEC_Tract') | (inputs == 'DEC_County'):
        # merge county name onto table
        df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
        df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
    
        
    print("Finished!")
    print("")


    print("Importing and compiling data from source")
    print("")

    # initialize empty list to store data frames
    list_df_acs = []
    
    # pull all years into one table (takes 2-3 minutes per year)
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_acs.append(
                    query_acs(api_Key      = api_key
                               , estimate  = estimate
                               , geography = geography
                               , variables = ','.join(dict_vars[str(year)])
                               , year      = year
                               , state     = state
                               , county    = dict_fips[state])
                )
                
            except:
                pass
                
    # combine all years and counties
    df_temp = pd.concat(list_df_acs)
    
    # left join data onto key for each geography type
    if inputs == 'DEC_Tract':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'left')
    if inputs == 'DEC_County':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')
    if inputs == 'DEC_MSA':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')
    
    print("Finished!")

## Data Cleaning

In [ ]:
# make copy
df_acs = df_acs_raw.copy()

if inputs == 'ACS_Tract':
    df_acs = df_acs.replace('-666666666', np.nan)
    df_acs = df_acs.replace('null', np.nan)

    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'tract', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'County Name', 'NAME', 'state', 
                     'county', 'tract', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    
    
    # Manually check column names and clean as needed
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
        , 'tract':'Tract ID'
    })

    if population_weights == 'Yes':
        df_pop = pd.read_csv(os.path.join(path_sp, 'Data', 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3', 'csv', 'Pop_3 Tract ACS5.csv'))

        conditions = [
                        (df_pop["Variable"] == 'All'                                            ),
                        (df_pop["Variable"] == 'American Indian or Alaska Native (NH)'         ),
                        (df_pop["Variable"] == 'Asian (NH)'                                     ),
                        (df_pop["Variable"] == 'Black or African American (NH)'                 ),
                        (df_pop["Variable"] == 'Hispanic or Latino'                             ),
                        (df_pop["Variable"] == 'Native Hawaiian or other Pacific Islander (NH)'),
                        (df_pop["Variable"] == 'White (NH)'                                     ),
                        (df_pop["Variable"] == 'Some other race (NH)'                           ),
                        (df_pop["Variable"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]

        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        
        df_acs = df_acs.merge(df_pop[['NAME', 'Year','Race_Ethnicity', 'Total']].rename(columns = {'Total':'Population'})
                              , on = ['NAME', 'Year', 'Race_Ethnicity'], how = 'left')
        


if inputs == 'ACS_County':
    
    df_acs = df_acs.replace('null', np.nan)
    df_acs = df_acs.dropna(axis = 0, how = "any")
    
    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'County Name', 'NAME', 'state', 
                       'county', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    
    
    # Manually check column names and clean as needed
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
    })



if inputs == 'ACS_MSA':
    
    df_acs = df_acs.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})
    
    # mapping
    df_msa_map = df_acs[df_acs['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates().rename(columns = {'NAME':'MSA'})
    df_acs = df_acs.merge(df_msa_map, on = 'MSA_ID')
    
    # Melt data from wide to long
    df_acs = pd.melt(df_acs
                      , id_vars = ['NAME', 'MSA', 'MSA_ID', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    
    # Convert imported values to numeric
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    
    
    # Merge label 2
    df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    
    
    # Subset
    df_acs = df_acs[['ID', 'Table Name', 'Label', 'MSA_ID', 'MSA', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]


if inputs == 'PUMS_PUMA':

    # Clean missing values
    for group in df_inputs['groups'].dropna().values:
        df_acs[group] = df_acs[group].astype(str).apply('{:0>2}'.format)

    df_acs['state'] = df_acs['state'].astype(str).apply('{:0>2}'.format)

    # Convert weighted column to integer
    df_acs[df_vars['Suggested Weight'].values[0]] = df_acs[df_vars['Suggested Weight'].values[0]].astype(int)


    # Import and merge Variable ID description
    df_acs[list(df_inputs['groups'].dropna().values)] = df_acs[list(df_inputs['groups'].dropna().values)].astype("string")

    # df_puma_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
    # df_puma_vars['Value1'] = df_puma_vars['Value1'].astype("string")
    df_vars['Value1'] = df_vars['Value1'].astype(str).apply('{:0>2}'.format)

    # df_puma_vars = df_puma_vars[df_puma_vars['Include'] == 'Yes']

    df_puma_vars = df_vars.pivot_table(index = ['Year', 'Value1']
                                           , columns = 'ID2'
                                           , values = 'Description2'
                                           , aggfunc = lambda x: x).reset_index()

    cols = ['Year', 'Value1'] + list(df_inputs['groups'].dropna().values)
    df_puma_vars = df_puma_vars[cols]

    # values
    list_values = []
    for group in df_inputs['groups'].dropna().values:
        list_values = list_values + list(df_acs[group].values)
    set_values = set(list_values)
    
    df_puma_vars = df_puma_vars[df_puma_vars['Value1'].isin(set_values)]
    df_puma_vars = df_puma_vars.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    for col in cols[2:]:
        df_acs = df_acs.merge(df_puma_vars[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
        df_acs = df_acs.drop(['Value1'], axis = 1)


df_acs.head()

## Organize Exports

In [ ]:
# Sort by census tract then by year then by race/ethnicity
df_acs['Race_Ethnicity_sort'] = pd.Categorical(df_acs['Race_Ethnicity'], ['All'
                                                                 , 'AMERICAN INDIAN AND ALASKA NATIVE ALONE'
                                                                 , 'ASIAN ALONE'
                                                                 , 'BLACK OR FRICAN AMERICAN ALONE'
                                                                 , 'HISPANIC OR LATINO'
                                                                 , 'NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE'
                                                                 , 'WHITE ALONE'
                                                                 , 'WHITE ALONE, NOT HISPANIC OR LATINO'
                                                                 , 'SOME OTHER RACE ALONE'
                                                                 , 'TWO OR MORE RACES'])


# sort and then remove categorical field
df_acs = df_acs.sort_values(by = ['NAME', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = True)
df_acs = df_acs.drop(['Race_Ethnicity_sort'], axis = 1)

In [ ]:
df_fips.head()

In [ ]:
# Create MPO and MSA groupings
df_mpo = df_fips[['County Name', 'MPO']]
df_msa = df_fips[['County Name', 'MSA']]


# Merge groupings
df_acs = df_acs.merge(df_mpo, on = ['County Name'], how = 'left')
df_acs = df_acs.merge(df_msa, on = ['County Name'], how = 'left')


# reorder columns
cols = ['Table ID', 'Table Name', 'Label', 'State FIPS', 'MPO', 'MSA', 'County Name', 
        'County FIPS', 'Tract ID', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total']
df_acs = df_acs[cols]


# missing values represent a population of 0
df_acs['Total'] = df_acs['Total'].fillna(0)


# view
df_acs.head()

In [ ]:
## Groupings roll up
df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 
                          'Variable', 'Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Total'].sum()
df_acs1['Proportion'] = df_acs1['Total'] / df_acs1[df_acs1['Variable'] != 'Total'].groupby(['NAME', 'Year'])['Total'].transform('sum')


# Counties
df_counties1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 
                               'Variable', 'Year', 'Race_Ethnicity'
                              ], as_index = False, sort = False)['Total'].sum()
df_counties1['Proportion'] = df_counties1['Total'] / df_counties1[df_counties1['Variable'] != 'Total'].groupby(['State FIPS', 'County FIPS', 'Year'])['Total'].transform('sum')


# MSA
df_msa1 = df_acs.groupby(['State FIPS', 'MSA', 
                          'Variable','Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Total'].sum()
df_msa1['Proportion'] = df_msa1['Total'] / df_msa1[df_msa1['Variable'] != 'Total'].groupby(['MSA', 'Year'])['Total'].transform('sum')


# MPO
df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                          'Variable', 'Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Total'].sum()
df_mpo1['Proportion'] = df_mpo1['Total'] / df_mpo1[df_mpo1['Variable'] != 'Total'].groupby(['MPO', 'Year'])['Total'].transform('sum')



# missing values represent a population of 0
df_acs1     ['Proportion'] = df_acs1     ['Proportion'].fillna(1)
df_counties1['Proportion'] = df_counties1['Proportion'].fillna(1)
df_msa1     ['Proportion'] = df_msa1     ['Proportion'].fillna(1)
df_mpo1     ['Proportion'] = df_mpo1     ['Proportion'].fillna(1)




# view
# df_acs1     .head()
# df_counties1.head()
# df_msa1     .head()
df_mpo1     .head()

In [ ]:
## Dcasts

# Tracts
df_acs2_pop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                       , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Total').reset_index()
df_acs2_prop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                       , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Proportion').reset_index()

# Counties
df_counties2_pop = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                        'County Name', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Total').reset_index()
df_counties2_prop = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                        'County Name', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Proportion').reset_index()

# MSA
df_msa2_pop = df_msa1.pivot_table(index = ['State FIPS', 'MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Total').reset_index()
df_msa2_prop = df_msa1.pivot_table(index = ['State FIPS', 'MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Proportion').reset_index()

# MPO
df_mpo2_pop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Total').reset_index()
df_mpo2_prop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Proportion').reset_index()


# missing values represent a population of 0
df_acs2_pop  = df_acs2_pop .fillna(0)
df_acs2_pop  = df_acs2_pop .fillna(0)
df_msa2_pop  = df_msa2_pop .fillna(0)
df_mpo2_pop  = df_mpo2_pop .fillna(0)
df_acs2_prop = df_acs2_prop.fillna(0)
df_acs2_prop = df_acs2_prop.fillna(0)
df_msa2_prop = df_msa2_prop.fillna(0)
df_mpo2_prop = df_mpo2_prop.fillna(0)


# view
df_msa2_prop.head(3)

#### Export

In [ ]:
# Set output name

name_output_long = ['ACS5 ', indicator_name, ' Long.xlsx']
name_output_wide = ['ACS5 ', indicator_name, ' Wide.xlsx']

name_output_long = "".join(name_output_long)
name_output_wide = "".join(name_output_wide)


In [ ]:
# Export long
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
    df_acs      .to_excel(writer, index = False, sheet_name = 'Full Tracts')
    df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
    df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
    df_msa1     .to_excel(writer, index = False, sheet_name = 'MSA'     )
    df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )

In [ ]:
# Export wide
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
    df_acs2_pop      .to_excel(writer, index = False, sheet_name = 'Tracts Total'   )
    df_counties2_pop .to_excel(writer, index = False, sheet_name = 'Counties Total' )
    df_msa2_pop      .to_excel(writer, index = False, sheet_name = 'MSA Total'      )
    df_mpo2_pop      .to_excel(writer, index = False, sheet_name = 'MPO Total'      )
    df_acs2_prop     .to_excel(writer, index = False, sheet_name = 'Tracts Prop'  )
    df_counties2_prop.to_excel(writer, index = False, sheet_name = 'Counties Prop')
    df_msa2_prop     .to_excel(writer, index = False, sheet_name = 'MSA Prop'     )
    df_mpo2_prop     .to_excel(writer, index = False, sheet_name = 'MPO Prop'     )